In [12]:
# import kagglehub

# # Download the dataset
# path = kagglehub.dataset_download("abhikjha/movielens-100k")

# print("Dataset downloaded to:", path)

In [13]:
import os
import pandas as pd
from math import sqrt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn import linear_model

### Explore the movie_lens 100k dataset to understand about recommendations
- Follow some online codes to understand the foundation

In [14]:
## Utilize schema from readme
u_data_cols = ['user_id', 'item_id', 'rating', 'timestamp']
u_item_cols = ['item_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
u_user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

In [15]:
df_users = pd.read_csv('data/u.user', sep='|', names=u_user_cols, encoding='latin-1')
df_items = pd.read_csv('data/u.item', sep='|', names=u_item_cols, encoding='latin-1')
df_ratings_base = pd.read_csv('data/ua.base', sep='\t', names=u_data_cols, encoding='latin-1')
df_ratings_test = pd.read_csv('data/ua.test', sep='\t', names=u_data_cols, encoding='latin-1')

In [16]:
df_users

,user_id,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213
...,...,...,...,...,...
938,939,26,F,student,33319
939,940,32,M,administrator,02215
940,941,20,M,student,97229
941,942,48,F,librarian,78209


In [17]:
rate_train = df_ratings_base.to_numpy()
rate_test = df_ratings_test.to_numpy()

### Item profile

In [18]:
X0 = df_items.to_numpy()
X_train_counts = X0[:, -19:]

In [19]:
from sklearn.feature_extraction.text import TfidfTransformer
transformer = TfidfTransformer(smooth_idf=True, norm ='l2')
tfidf = transformer.fit_transform(X_train_counts.tolist()).toarray()

In [20]:
tfidf

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.53676706, 0.65097024, ..., 0.53676706, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [21]:
import numpy as np
def get_items_rated_by_user(rate_matrix, user_id):
    """
    in each line of rate_matrix, we have infor: user_id, item_id, rating (scores), time_stamp
    we care about the first three values
    return (item_ids, scores) rated by user user_id
    """
    y = rate_matrix[:,0] # all users
    # item indices rated by user_id
    # we need to +1 to user_id since in the rate_matrix, id starts from 1 
    # while index in python starts from 0
    ids = np.where(y == user_id +1)[0] 
    item_ids = rate_matrix[ids, 1] - 1 # index starts from 0 
    scores = rate_matrix[ids, 2]
    return (item_ids, scores)

In [22]:
from sklearn.linear_model import Ridge
from sklearn import linear_model

n_users = df_users.shape[0]  # or len(df_users)

d = tfidf.shape[1] # data dimension
W = np.zeros((d, n_users))
b = np.zeros((1, n_users))


In [23]:
W.shape

(19, 943)

In [24]:
b.shape

(1, 943)

In [25]:

for n in range(n_users):    
    ids, scores = get_items_rated_by_user(rate_train, n)
    
    if len(ids) == 0:
        continue
        
    clf = Ridge(alpha=0.01, fit_intercept=True)
    Xhat = tfidf[ids, :]
    
    clf.fit(Xhat, scores) 
    W[:, n] = clf.coef_.ravel()
    b[0, n] = clf.intercept_

In [26]:
Yhat = tfidf.dot(W) + b
Yhat

array([[2.91838332, 3.99617751, 1.90305533, ..., 5.03731383, 4.77954355,
        3.16575325],
       [2.80263057, 3.43501556, 3.29904303, ..., 1.09956555, 4.75937828,
        3.69972213],
       [3.4830386 , 1.56190995, 1.26948894, ..., 7.35604832, 4.18870316,
        3.80591695],
       ...,
       [4.12679785, 4.09647029, 3.02107854, ..., 9.53834396, 4.64249022,
        3.27349356],
       [3.56723527, 3.45068283, 2.92544446, ..., 3.68357866, 3.6551852 ,
        2.75740298],
       [4.13381936, 3.80361821, 3.09903449, ..., 9.53834396, 4.41870642,
        3.93041386]])

In [27]:
n = 10
np.set_printoptions(precision=2) # 2 digits after . 
ids, scores = get_items_rated_by_user(rate_test, n)
Yhat[n, ids]
print('Rated movies ids :', ids )
print('True ratings     :', scores)
print('Predicted ratings:', Yhat[ids, n])

Rated movies ids : [ 37 109 110 226 424 557 722 724 731 739]
True ratings     : [3 3 4 3 4 3 5 3 3 4]
Predicted ratings: [3.18 3.13 3.42 3.09 3.35 5.2  4.01 3.35 3.42 3.72]


In [28]:
from math import sqrt

def evaluate(Yhat, rates, W, b):
    se = 0
    cnt = 0
    # Safely get the number of users from the rating matrix shape (columns)
    n_users = rates.shape[1]
    
    for n in range(n_users):  # Python 3 uses range instead of xrange
        ids, scores_truth = get_items_rated_by_user(rates, n)
        scores_pred = Yhat[ids, n]
        e = scores_truth - scores_pred 
        se += (e * e).sum(axis=0)
        cnt += e.size 
        
    return sqrt(se / cnt)

# Python 3 print function requires parentheses
print('RMSE for training:', evaluate(Yhat, rate_train, W, b))
print('RMSE for test    :', evaluate(Yhat, rate_test, W, b))

RMSE for training: 1.0045223804032835
RMSE for test    : 1.6796393345565312


### Question: Is there the vice-versa method existed ? 
Mean loop through each item and learn the pattern   
-> It called demographic filtering
-> Purpose: recommend movies for similar customers (using demographic info)   
-> Use case: Cold-start when customers only input basic demographic info and the service can recommend some items based on customer-profile 

In [29]:
# ==========================================
# STEP 1: Preprocess User Demographics
# ==========================================

# Copy demographics and drop zip_code (too sparse to be useful)
df_demographics = df_users[["user_id", "age", "gender", "occupation"]].copy()

# One-hot encode categorical features (gender and occupation)
df_demographics = pd.get_dummies(
    df_demographics, columns=["gender", "occupation"], drop_first=True
)

# Scale Age using Min-Max scaling to keep it between 0 and 1
max_age = df_demographics["age"].max()
min_age = df_demographics["age"].min()
df_demographics["age"] = (df_demographics["age"] - min_age) / (
    max_age - min_age
)

# Convert to NumPy array aligned with 0-based user indices
df_demographics = df_demographics.sort_values("user_id")
U_demographics = (
    df_demographics.drop(columns=["user_id"]).to_numpy().astype(float)
)


In [30]:
U_demographics

array([[0.26, 1.  , 0.  , ..., 0.  , 1.  , 0.  ],
       [0.7 , 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.24, 1.  , 0.  , ..., 0.  , 0.  , 1.  ],
       ...,
       [0.2 , 1.  , 0.  , ..., 1.  , 0.  , 0.  ],
       [0.62, 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.23, 1.  , 0.  , ..., 1.  , 0.  , 0.  ]])

In [31]:
# ==========================================
# STEP 2: Initialize Matrices
# ==========================================
rate_train = df_ratings_base.to_numpy()
rate_test = df_ratings_test.to_numpy()

n_users = df_users.shape[0]
n_items = df_items.shape[0]
d = U_demographics.shape[1]  # Number of demographic features

# Initialize Weights (V) and Biases (c) to 0
V = np.zeros((d, n_items))
c = np.zeros((1, n_items))  # Set directly to 0 instead of global_mean


In [32]:
V.shape

(22, 1682)

In [33]:
c

array([[0., 0., 0., ..., 0., 0., 0.]])

In [34]:
# ==========================================
# STEP 3: Train a Model for Each Movie
# ==========================================
for i in range(n_items):
    # Find all users who rated movie (i + 1)
    ids = np.where(rate_train[:, 1] == i + 1)[0]

    # Skip if the movie has too few ratings to fit a model stably
    if len(ids) < 5:
        continue

    # Get the 0-based user indices and their ratings
    user_ids = rate_train[ids, 0] - 1
    scores = rate_train[ids, 2]

    # Get the demographic vectors for those users
    Xhat = U_demographics[user_ids, :]

    # Fit Ridge Regression using demographics to predict the rating
    clf = Ridge(alpha=1.0, fit_intercept=True)
    clf.fit(Xhat, scores)

    # Save the parameters
    V[:, i] = clf.coef_.ravel()
    c[0, i] = clf.intercept_



In [37]:
V

array([[-3.52e-01, -3.85e-02, -9.27e-01, ...,  0.00e+00,  0.00e+00,
         0.00e+00],
       [-7.41e-04, -3.16e-01,  4.09e-01, ...,  0.00e+00,  0.00e+00,
         0.00e+00],
       [ 2.34e-01, -1.39e-01,  0.00e+00, ...,  0.00e+00,  0.00e+00,
         0.00e+00],
       ...,
       [-2.41e-01, -2.03e-01,  1.88e-02, ...,  0.00e+00,  0.00e+00,
         0.00e+00],
       [-7.32e-04, -4.91e-01,  9.44e-01, ...,  0.00e+00,  0.00e+00,
         0.00e+00],
       [-3.63e-01, -1.96e-01, -9.30e-01, ...,  0.00e+00,  0.00e+00,
         0.00e+00]])

In [38]:
np.save('data/V_matrix.npy', V)

In [35]:
# ==========================================
# STEP 4: Prediction (Inference)
# ==========================================
# Predict ratings: shape is (n_users, n_items)
Y_pred = U_demographics.dot(V) + c

# Transpose to shape (n_items, n_users) to match your evaluation function
Yhat = Y_pred.T

# ==========================================
# STEP 5: Evaluation
# ==========================================
def get_items_rated_by_user(rate_matrix, user_id):
    y = rate_matrix[:, 0]
    ids = np.where(y == user_id + 1)[0]
    item_ids = rate_matrix[ids, 1] - 1
    scores = rate_matrix[ids, 2]
    return (item_ids, scores)


def evaluate(Yhat, rates):
    se = 0
    cnt = 0
    n_users = rates.shape[1] if len(rates.shape) > 1 else len(rates)

    for n in range(n_users):
        ids, scores_truth = get_items_rated_by_user(rates, n)
        if len(ids) == 0:
            continue
        scores_pred = Yhat[ids, n]
        e = scores_truth - scores_pred
        se += (e * e).sum(axis=0)
        cnt += e.size

    return sqrt(se / cnt)


In [36]:
# Print results
print("Demographic Training RMSE:", evaluate(Yhat, rate_train))
print("Demographic Test RMSE    :", evaluate(Yhat, rate_test))

Demographic Training RMSE: 0.8540116348118488
Demographic Test RMSE    : 1.1086996762722534
